<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 10</h2>
<h2>AES (Advanced Encryption Standard)</h2>
<h3 style="color: #8B4513;">Rijndael - müasir kriptoqrafiyanın qızıl standartı</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: AES haqqında (10 dəq)](#xatırlatma-aes-haqqında-10-dəq)
5. [$GF(2^8)$ arifmetikası (20 dəq)](#gf2-8-arifmetikası-20-dəq)
6. [AES S-qutusu (SubBytes) (15 dəq)](#aes-s-qutusu-subbytes-15-dəq)
7. [ShiftRows (15 dəq)](#shiftrows-15-dəq)
8. [MixColumns (25 dəq)](#mixcolumns-25-dəq)
9. [Açar genişləndirmə (Key Expansion) (20 dəq)](#açar-genişləndirmə-key-expansion-20-dəq)
10. [Tam AES raundu (20 dəq)](#tam-aes-raundu-20-dəq)
11. [PyCryptodome ilə AES (15 dəq)](#pycryptodome-ilə-aes-15-dəq)
12. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
13. [Ev tapşırığı](#ev-tapşırığı)
14. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
15. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ AES-in tarixi və əhəmiyyətini başa düşəcəksiniz  
✅ AES-in SPN quruluşunu izah edə biləcəksiniz  
✅ AES-in dörd əsas mərhələsini (SubBytes, ShiftRows, MixColumns, AddRoundKey) implementasiya edə biləcəksiniz  
✅ $GF(2^8)$ arifmetikasını başa düşəcəksiniz  
✅ PyCryptodome kitabxanası ilə AES istifadə edə biləcəksiniz  
✅ AES-in CBC, CTR kimi rejimlərini tətbiq edə biləcəksiniz  
✅ AES-128, AES-192, AES-256 arasındakı fərqləri izah edə biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| PyCryptodome | `!pip install pycryptodome` | AES dəstəyi və `Crypto.Random.get_random_bytes` ilə kriptoqrafik təsadüfilik |
| Python standart kitabxanası | `os`, `secrets` (daxili) | Fayl əməliyyatları və kriptoqrafik təhlükəsiz təsadüfi ədədlər |

> 💡 **Qeyd:** PyCryptodome kitabxanası mütləq quraşdırılmalıdır. Bu kitabxana olmadan AES-in real implementasiyasını test edə bilməyəcəksiniz. Kriptoqrafik açarlar, IV-lər və nonce-lar üçün `random` modulundan istifadə etməyin; bunun əvəzinə `Crypto.Random.get_random_bytes`, `secrets` və ya `os.urandom` istifadə edin.

In [ ]:
# Lazımi kitabxanaların quraşdırılması

import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "pycryptodome", "--quiet"])

print("✅ Kitabxanalar quraşdırıldı")


## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import os
import random
import math
import time
from collections import Counter

# PyCryptodome kitabxanası
try:
    from Crypto.Cipher import AES
    from Crypto.Random import get_random_bytes
    from Crypto.Util.Padding import pad, unpad
    from Crypto.Protocol.KDF import PBKDF2
    print("✅ PyCryptodome yüklü - AES istifadə edilə bilər")
    AES_AVAILABLE = True
except ImportError:
    print("❌ PyCryptodome yoxdur! Əvvəlcə quraşdırın: pip install pycryptodome")
    AES_AVAILABLE = False
    AES = None

print(f"\n🐍 Python versiyası: {sys.version}")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
import os
from pathlib import Path

# İşçi qovluğu yaradaq
workspace_parts = ("crypto_workshop", "lecture10")
workspace_name = Path(*workspace_parts)

base_dir_env = os.environ.get("CRYPTO_WORKSHOP_BASE_DIR")
if base_dir_env:
    base_dir = Path(base_dir_env).resolve()
else:
    cwd = Path.cwd().resolve()
    parts = cwd.parts
    base_dir = cwd

    # Əgər bu xana artıq işçi qovluğun içindən yenidən işə salınıbsa,
    # yolu yenidən iç-içə yaratmamaq üçün ilkin baza qovluğunu tapırıq.
    for i in range(len(parts) - len(workspace_parts) + 1):
        if tuple(parts[i:i + len(workspace_parts)]) == workspace_parts:
            base_dir = Path(*parts[:i])
            break

    os.environ["CRYPTO_WORKSHOP_BASE_DIR"] = str(base_dir)

workspace = base_dir / workspace_name
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {os.getcwd()}")
print(f"📂 Qovluğun məzmunu: {os.listdir('.')}")

## 📖 Xatırlatma: AES haqqında (10 dəq)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 AES-in yaranması (1997-2001)</h4>
<p>1997-ci ildə NIST yeni şifrləmə standartı üçün açıq müsabiqə elan etdi. 15 ölkədən 15 namizəd alqoritm təqdim edildi. 5 finalçı (MARS, RC6, Rijndael, Serpent, Twofish) arasından <b>Rijndael</b> seçildi. 2001-ci ildə <b>FIPS 197</b> standartı kimi qəbul edildi.</p>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🔑 AES xüsusiyyətləri</h4>
<ul>
    <li><b>Blok ölçüsü:</b> 128-bit (həmişə)</li>
    <li><b>Açar uzunluqları:</b> 128, 192, 256-bit</li>
    <li><b>Raund sayı:</b> 10 (AES-128), 12 (AES-192), 14 (AES-256)</li>
    <li><b>Quruluş:</b> SPN (Substitution-Permutation Network)</li>
    <li><b>Dövlət (state):</b> 4×4 bayt matrisi</li>
</ul>
</div>

![AES strukturu](https://upload.wikimedia.org/wikipedia/commons/thumb/5/50/AES_%28Rijndael%29_Round_Function.png/600px-AES_%28Rijndael%29_Round_Function.png)

## 🧮 $GF(2^8)$ arifmetikası (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔢 GF(2⁸) meydanı</h4>
<p>AES $GF(2^8)$ meydanında işləyir və aşağıdakı ayrılmaz çoxhədlidən istifadə edir:</p>
<p style="text-align: center; font-size: 1.2em;">$$m(x) = x^8 + x^4 + x^3 + x + 1$$</p>
<p><b>Xüsusiyyətlər:</b></p>
<ul>
    <li>Toplama = XOR (sürətli, sadə)</li>
    <li>Vurma = xtime əsasında (mürəkkəb, lakin sürətli)</li>
    <li>Hər bir bayt (0-255) bu meydanda bir ədəddir</li>
</ul>
</div>

In [ ]:
def gf_multiply(a, b):
    """
    GF(2^8)-də vurma əməliyyatı
    m(x) = x^8 + x^4 + x^3 + x + 1 (0x11B)

    Parametrlər:
        a, b: 0-255 arası ədədlər

    Qaytarır:
        a * b (mod m(x))
    """
    result = 0
    while b > 0:
        if b & 1:
            result ^= a
        a <<= 1
        if a & 0x100:  # əgər 9-cu bit varsa
            a ^= 0x11B   # m(x) ilə mod
        b >>= 1
    return result & 0xFF

def xtime(x):
    """
    xtime(x) = 02 * x (GF(2^8)-də)
    AES-də tez-tez istifadə olunur
    """
    x <<= 1
    if x & 0x100:
        x ^= 0x11B
    return x & 0xFF

def gf_pow(a, n):
    """
    GF(2^8)-də a^n hesablayır
    """
    result = 1
    for _ in range(n):
        result = gf_multiply(result, a)
    return result

def gf_inverse(a):
    """
    GF(2^8)-də a^-1 tapır (a != 0)
    Fermat teoremi: a^(254) = a^-1
    """
    if a == 0:
        return 0
    return gf_pow(a, 254)

# Test
print("=" * 70)
print("🧮 GF(2⁸) ARİFMETİKASI TESTİ")
print("=" * 70)

# xtime testi
print(f"xtime(0x57) = 0x{xtime(0x57):02x} (gözlənilən: 0xAE)")
print(f"xtime(0x83) = 0x{xtime(0x83):02x}")

# Vurma testi
print(f"\n0x57 * 0x13 = 0x{gf_multiply(0x57, 0x13):02x} (gözlənilən: 0xFE)")
print(f"0x02 * 0x87 = 0x{gf_multiply(0x02, 0x87):02x}")
print(f"0x03 * 0x6E = 0x{gf_multiply(0x03, 0x6E):02x}")

# Tərs element
print(f"\n0x03-ün tərsi: 0x{gf_inverse(0x03):02x} (03 * DB = {gf_multiply(0x03, gf_inverse(0x03)):02x})")

### ✍️ Çalışma 1: $GF(2^8)$ əməliyyatları (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **xtime funksiyası:** xtime funksiyasını istifadə edərək aşağıdakı dəyərləri hesablayın:
   * xtime(0x83)
   * xtime(0xAB)
   * xtime(0x00)

2. **gf_multiply funksiyası:** gf_multiply funksiyasını istifadə edərək aşağıdakıları hesablayın:
   * 0x02 * 0x87
   * 0x03 * 0x6E
   * 0x09 * 0xA6

3. **Nəzəri sual:** Niyə $GF(2^8)$-də toplama əməliyyatı sadəcə XOR-dur? Bunun səbəbini izah edin.

4. **Tərs element:** gf_inverse funksiyasından istifadə edərək aşağıdakı baytların tərsini tapın:
   * 0x01
   * 0x02
   * 0xFF

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. xtime funksiyası
print("\n1. XTIME FUNKSİYASI:")
print(f"   xtime(0x83) = 0x{xtime(0x83):02x}")
print(f"   xtime(0xAB) = 0x{xtime(0xAB):02x}")
print(f"   xtime(0x00) = 0x{xtime(0x00):02x}")

# 2. gf_multiply funksiyası
print("\n2. GF_MULTIPLY FUNKSİYASI:")
print(f"   0x02 * 0x87 = 0x{gf_multiply(0x02, 0x87):02x}")
print(f"   0x03 * 0x6E = 0x{gf_multiply(0x03, 0x6E):02x}")
print(f"   0x09 * 0xA6 = 0x{gf_multiply(0x09, 0xA6):02x}")

# 3. Nəzəri sual
print("\n3. NƏZƏRİ SUAL:")
print("""
   GF(2⁸)-də toplama XOR-dur, çünki:
   • GF(2) meydanında toplama XOR-dur (1+1=0, 1+0=1)
   • GF(2⁸) bu meydanın genişləndirilməsidir
   • Polinomların əmsalları GF(2)-dədir, ona görə toplama əmsal-əmsal XOR deməkdir
   • Bu, çox sürətli hardware implementasiyasına imkan verir
""")

# 4. Tərs element
print("\n4. TƏRS ELEMENT:")
print(f"   0x01-in tərsi: 0x{gf_inverse(0x01):02x}")
print(f"   0x02-in tərsi: 0x{gf_inverse(0x02):02x}")
print(f"   0xFF-in tərsi: 0x{gf_inverse(0xFF):02x}")

## 🔢 AES S-qutusu (SubBytes) (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔀 SubBytes - Qeyri-xətti əvəzetmə</h4>
<p>AES S-qutusu iki mərhələdə qurulur:</p>
<ol>
    <li>GF(2⁸)-də tərs elementin tapılması ($a \rightarrow a^{-1}$, 0x00 → 0x00)</li>
    <li>Affin transformasiya: $b_i' = b_i \oplus b_{(i+4)\ mod\ 8} \oplus b_{(i+5)\ mod\ 8} \oplus b_{(i+6)\ mod\ 8} \oplus b_{(i+7)\ mod\ 8} \oplus c_i$</li>
</ol>
</div>

In [ ]:
def aes_sbox():
    """
    AES S-qutusunu qaytarır (16x16 cədvəl)
    FIPS 197 standartından götürülmüşdür
    """
    # AES S-qutusu (FIPS 197-dən)
    sbox = [
        [0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76],
        [0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0],
        [0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15],
        [0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75],
        [0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84],
        [0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf],
        [0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8],
        [0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2],
        [0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73],
        [0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb],
        [0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79],
        [0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08],
        [0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a],
        [0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e],
        [0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf],
        [0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16]
    ]
    return sbox

def aes_inv_sbox():
    """
    AES tərs S-qutusunu qaytarır
    """
    sbox = aes_sbox()
    inv_sbox = [[0]*16 for _ in range(16)]

    for i in range(16):
        for j in range(16):
            val = sbox[i][j]
            inv_sbox[val >> 4][val & 0xF] = (i << 4) | j

    return inv_sbox

def sub_bytes(byte, sbox):
    """
    Bir bayta S-qutusunu tətbiq edir
    """
    row = (byte >> 4) & 0xF
    col = byte & 0xF
    return sbox[row][col]

def sub_bytes_state(state, sbox):
    """
    Bütün dövlətə (4x4 matris) S-qutusunu tətbiq edir
    """
    result = []
    for i in range(4):
        row = []
        for j in range(4):
            row.append(sub_bytes(state[i][j], sbox))
        result.append(row)
    return result

# Test
sbox = aes_sbox()
inv_sbox = aes_inv_sbox()

print("\n" + "=" * 70)
print("🔢 AES S-QUTUSU TESTİ")
print("=" * 70)

test_byte = 0x53  # 'S' hərfi
result = sub_bytes(test_byte, sbox)
print(f"S-box(0x{test_byte:02x}) = 0x{result:02x} (gözlənilən: 0xed)")

# Tərs test
inv_result = sub_bytes(result, inv_sbox)
print(f"InvS-box(0x{result:02x}) = 0x{inv_result:02x}")
print(f"Orijinala bərabər? {test_byte == inv_result}")

### ✍️ Çalışma 2: S-qutusu (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **S-qutusu axtarışı:** S-qutusundan istifadə edərək aşağıdakı baytları əvəz edin:
   * 0x00
   * 0xFF
   * 0x2B
   * 0x7E

2. **Tərs S-qutusu:** inv_sbox cədvəlindən istifadə edərək yuxarıdakı nəticələri orijinala çevirin.

3. **Nəzəri sual:** Niyə S-qutusunda heç bir sabit nöqtə yoxdur? ($S(a) = a$ şərtini ödəyən a yoxdur)

4. **Affin transformasiya:** S-qutusunun affin transformasiya hissəsini izah edin. Bu, nə üçün vacibdir?

In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

# 1. S-qutusu axtarışı
print("\n1. S-QUTUSU AXTARIŞI:")
test_bytes = [0x00, 0xFF, 0x2B, 0x7E]
results = []
for b in test_bytes:
    r = sub_bytes(b, sbox)
    results.append(r)
    print(f"   S-box(0x{b:02x}) = 0x{r:02x}")

# 2. Tərs S-qutusu
print("\n2. TƏRS S-QUTUSU:")
for orig, res in zip(test_bytes, results):
    inv = sub_bytes(res, inv_sbox)
    print(f"   InvS-box(0x{res:02x}) = 0x{inv:02x} - {'✓' if orig == inv else '✗'}")

# 3. Nəzəri sual
print("\n3. NƏZƏRİ SUAL:")
print("""
   S-qutusunda sabit nöqtə yoxdur, çünki:
   • Tərs element + affin transformasiya kombinasiyası
   • Affin transformasiya xüsusi seçilib ki, S(a)=a olmasın
   • Bu, xətti kriptoanalizə qarşı davamlılıq üçün vacibdir
   • Yalnız 0x63 və 0x63 arasında əlaqə var (S(0x63)=0x63 deyil)
""")

# 4. Affin transformasiya
print("\n4. AFFİN TRANSFORMASİYA:")
print("""
   Affin transformasiya düsturu:
   b_i' = b_i ⊕ b_(i+4) ⊕ b_(i+5) ⊕ b_(i+6) ⊕ b_(i+7) ⊕ c_i

   Əhəmiyyəti:
   • GF(2^8)-də multiplikativ tərs alma S-qutusunun qeyri-xətti hissəsidir
   • Affin hissə xətti transformasiya + sabitdən ibarətdir; özü qeyri-xəttilik mənbəyi deyil
   • XOR və bit döndərmə ilə asan implementasiya olunur
   • Sabit c_i (0x63) sabit və əks-sabit nöqtələrdən yayınmağa kömək edir
""")

## 🔄 ShiftRows (15 dəq)

In [ ]:
def shift_rows(state):
    """
    ShiftRows əməliyyatı
    state: 4x4 matris (list of lists)
    """
    result = [row[:] for row in state]  # kopiya

    # 2-ci sətir: 1 bayt sola
    result[1][0], result[1][1], result[1][2], result[1][3] = \
        state[1][1], state[1][2], state[1][3], state[1][0]

    # 3-cü sətir: 2 bayt sola
    result[2][0], result[2][1], result[2][2], result[2][3] = \
        state[2][2], state[2][3], state[2][0], state[2][1]

    # 4-cü sətir: 3 bayt sola (1 bayt sağa)
    result[3][0], result[3][1], result[3][2], result[3][3] = \
        state[3][3], state[3][0], state[3][1], state[3][2]

    return result

def inv_shift_rows(state):
    """
    ShiftRows-un tərsi (deşifrləmə üçün)
    """
    result = [row[:] for row in state]  # kopiya

    # 2-ci sətir: 1 bayt sağa
    result[1][0], result[1][1], result[1][2], result[1][3] = \
        state[1][3], state[1][0], state[1][1], state[1][2]

    # 3-cü sətir: 2 bayt sağa
    result[2][0], result[2][1], result[2][2], result[2][3] = \
        state[2][2], state[2][3], state[2][0], state[2][1]

    # 4-cü sətir: 3 bayt sağa (1 bayt sola)
    result[3][0], result[3][1], result[3][2], result[3][3] = \
        state[3][1], state[3][2], state[3][3], state[3][0]

    return result

# Test
print("\n" + "=" * 70)
print("🔄 SHIFTROWS TESTİ")
print("=" * 70)

test_state = [
    [0x00, 0x01, 0x02, 0x03],
    [0x10, 0x11, 0x12, 0x13],
    [0x20, 0x21, 0x22, 0x23],
    [0x30, 0x31, 0x32, 0x33]
]

print("Orijinal dövlət:")
for row in test_state:
    print(f"  {[f'{x:02x}' for x in row]}")

shifted = shift_rows(test_state)
print("\nShiftRows sonrası:")
for row in shifted:
    print(f"  {[f'{x:02x}' for x in row]}")

inv_shifted = inv_shift_rows(shifted)
print("\nInvShiftRows sonrası:")
for row in inv_shifted:
    print(f"  {[f'{x:02x}' for x in row]}")
print(f"\nOrijinala bərabər? {inv_shifted == test_state}")

### ✍️ Çalışma 3: ShiftRows (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **ShiftRows tətbiqi:** Verilmiş dövlət üçün ShiftRows tətbiq edin:
   $$
   \begin{bmatrix}
   0x01 & 0x02 & 0x03 & 0x04 \\
   0x05 & 0x06 & 0x07 & 0x08 \\
   0x09 & 0x0A & 0x0B & 0x0C \\
   0x0D & 0x0E & 0x0F & 0x10
   \end{bmatrix}
   $$

2. **Tərsini yoxlama:** ShiftRows tətbiq edildikdən sonra `inv_shift_rows` ilə orijinalı bərpa edin.

3. **Yayılma rolu:** ShiftRows yayılmanı (diffusion) necə təmin edir? SubBytes-dən sonra gəlməsi nə üçün vacibdir?

4. **Səbəb sual:** Niyə hər sətir fərqli sayda sürüşdürülür? 4-cü sətir niyə 3 dəfə sürüşdürülür?


In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

# 1. ShiftRows tətbiqi
print("\n1. SHIFTROWS TƏTBİQİ:")
state1 = [
    [0x01, 0x02, 0x03, 0x04],
    [0x05, 0x06, 0x07, 0x08],
    [0x09, 0x0A, 0x0B, 0x0C],
    [0x0D, 0x0E, 0x0F, 0x10]
]

print("Orijinal:")
for row in state1:
    print(f"  {[f'{x:02x}' for x in row]}")

shifted1 = shift_rows(state1)
print("\nShiftRows sonrası:")
for row in shifted1:
    print(f"  {[f'{x:02x}' for x in row]}")

# 2. Tərsini yoxlama
print("\n2. TƏRSİNİ YOXLAMA:")
inv_shifted1 = inv_shift_rows(shifted1)
print("InvShiftRows sonrası:")
for row in inv_shifted1:
    print(f"  {[f'{x:02x}' for x in row]}")
print(f"\n   Orijinala bərabər? {inv_shifted1 == state1}")

# 3. Yayılma rolu
print("\n3. YAYILMA ROLU:")
print("""
   ShiftRows yayılmaya (diffusion) kömək edir:
   • SubBytes hər baytı müstəqil dəyişir
   • ShiftRows hər sətiri fərqli miqdarda sola sürüşdürərək baytları sütunlar arasında yerini dəyişir
   • Növbəti MixColumns addımı buna görə müxtəlif ilkin sütunlardan gələn baytları birlikdə qarışdırır
   • Bu təsir bir neçə raund ərzində dəyişikliyin bütün state üzrə yayılmasına imkan verir
""")

# 4. Səbəb sual
print("\n4. SƏBƏB SUAL:")
print("""
   Fərqli sürüşdürmə miqdarları:
   • 0,1,2,3 bayt sürüşdürmə hər sətiri fərqli sütun düzülüşünə aparır
   • 4-cü sətir 3 bayt sürüşdürülür, çünki 4 bayt sürüşdürmə orijinal düzülüşə bərabər olardı
   • Əsas məqsəd sadəcə inversiya və ya asan deşifrləmə deyil; invertivlik zəruridir, amma çoxlu permutasiyalar invertiv ola bilərdi
   • Bu ofsetlər MixColumns ilə birlikdə baytların sonrakı raundlarda state boyunca yayılmasını təmin edən Rijndael dizayn parametrləridir
""")


## 🔀 MixColumns (25 dəq)

In [ ]:
def mix_columns(state):
    """
    MixColumns əməliyyatı
    Hər bir sütuna matris vurması tətbiq edir
    """
    result = [[0]*4 for _ in range(4)]

    for c in range(4):
        # Sütun c
        s0 = state[0][c]
        s1 = state[1][c]
        s2 = state[2][c]
        s3 = state[3][c]

        # [02 03 01 01] matrisi ilə vurma
        result[0][c] = gf_multiply(0x02, s0) ^ gf_multiply(0x03, s1) ^ s2 ^ s3
        result[1][c] = s0 ^ gf_multiply(0x02, s1) ^ gf_multiply(0x03, s2) ^ s3
        result[2][c] = s0 ^ s1 ^ gf_multiply(0x02, s2) ^ gf_multiply(0x03, s3)
        result[3][c] = gf_multiply(0x03, s0) ^ s1 ^ s2 ^ gf_multiply(0x02, s3)

    return result

def inv_mix_columns(state):
    """
    MixColumns-un tərsi (deşifrləmə üçün)
    [0E 0B 0D 09] matrisi ilə vurma
    """
    result = [[0]*4 for _ in range(4)]

    for c in range(4):
        s0 = state[0][c]
        s1 = state[1][c]
        s2 = state[2][c]
        s3 = state[3][c]

        # [0E 0B 0D 09] matrisi ilə vurma
        result[0][c] = (gf_multiply(0x0E, s0) ^
                        gf_multiply(0x0B, s1) ^
                        gf_multiply(0x0D, s2) ^
                        gf_multiply(0x09, s3))
        result[1][c] = (gf_multiply(0x09, s0) ^
                        gf_multiply(0x0E, s1) ^
                        gf_multiply(0x0B, s2) ^
                        gf_multiply(0x0D, s3))
        result[2][c] = (gf_multiply(0x0D, s0) ^
                        gf_multiply(0x09, s1) ^
                        gf_multiply(0x0E, s2) ^
                        gf_multiply(0x0B, s3))
        result[3][c] = (gf_multiply(0x0B, s0) ^
                        gf_multiply(0x0D, s1) ^
                        gf_multiply(0x09, s2) ^
                        gf_multiply(0x0E, s3))

    return result

# Test
print("\n" + "=" * 70)
print("🔀 MIXCOLUMNS TESTİ")
print("=" * 70)

test_state_mix = [
    [0x87, 0x6E, 0x46, 0xA6],
    [0xF2, 0x4C, 0xE7, 0x8C],
    [0x4D, 0x90, 0x4A, 0xD8],
    [0x97, 0xEC, 0xC3, 0x95]
]

print("Orijinal dövlət (ilk sütun):")
print(f"  [{test_state_mix[0][0]:02x}, {test_state_mix[1][0]:02x}, "
      f"{test_state_mix[2][0]:02x}, {test_state_mix[3][0]:02x}]")

mixed = mix_columns(test_state_mix)
print("\nMixColumns sonrası (ilk sütun):")
print(f"  [{mixed[0][0]:02x}, {mixed[1][0]:02x}, "
      f"{mixed[2][0]:02x}, {mixed[3][0]:02x}]")

inv_mixed = inv_mix_columns(mixed)
print("\nInvMixColumns sonrası (ilk sütun):")
print(f"  [{inv_mixed[0][0]:02x}, {inv_mixed[1][0]:02x}, "
      f"{inv_mixed[2][0]:02x}, {inv_mixed[3][0]:02x}]")
print(f"\nOrijinala bərabər? {inv_mixed == test_state_mix}")

### ✍️ Çalışma 4: MixColumns (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Əl ilə hesablama:** $[0x87, 0x6E, 0x46, 0xA6]$ sütunu üçün MixColumns əməliyyatını əl ilə hesablayın.

2. **Proqram testi:** mix_columns funksiyasını test edin və nəticələri yoxlayın.

3. **Tərs funksiya:** inv_mix_columns funksiyasının mix_columns-un tərsi olduğunu yoxlayın.

4. **Yayılma rolu:** Niyə MixColumns yayılma (diffusion) üçün vacibdir? Bir bayt dəyişikliyi neçə bayta təsir edir?

5. **Matris seçimi:** Niyə məhz [02 03 01 01] matrisi seçilib? Bu matrisin xüsusiyyətləri nələrdir?

In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

# 1-2. Əl ilə hesablama və proqram testi
print("\n1-2. MİXCOLUMNS HESABLAMASI:")
s0, s1, s2, s3 = 0x87, 0x6E, 0x46, 0xA6

t0 = gf_multiply(0x02, s0) ^ gf_multiply(0x03, s1) ^ s2 ^ s3
t1 = s0 ^ gf_multiply(0x02, s1) ^ gf_multiply(0x03, s2) ^ s3
t2 = s0 ^ s1 ^ gf_multiply(0x02, s2) ^ gf_multiply(0x03, s3)
t3 = gf_multiply(0x03, s0) ^ s1 ^ s2 ^ gf_multiply(0x02, s3)

print(f"   Giriş: [{s0:02x}, {s1:02x}, {s2:02x}, {s3:02x}]")
print(f"   Çıxış: [{t0:02x}, {t1:02x}, {t2:02x}, {t3:02x}]")

# 3. Tərs funksiya
print("\n3. TƏRS FUNKSİYA:")
state2 = [[0x87, 0,0,0], [0x6E,0,0,0], [0x46,0,0,0], [0xA6,0,0,0]]
mixed2 = mix_columns(state2)
inv_mixed2 = inv_mix_columns(mixed2)
print(f"   Orijinal: [{state2[0][0]:02x}, {state2[1][0]:02x}, "
      f"{state2[2][0]:02x}, {state2[3][0]:02x}]")
print(f"   Bərpa:    [{inv_mixed2[0][0]:02x}, {inv_mixed2[1][0]:02x}, "
      f"{inv_mixed2[2][0]:02x}, {inv_mixed2[3][0]:02x}]")

# 4. Yayılma rolu
print("\n4. YAYILMA ROLU:")
print("""
   MixColumns hər bir sütunu qarışdırır:
   • Bir bayt dəyişikliyi bütün sütuna təsir edir
   • ShiftRows ilə birlikdə 2 raundda tam yayılma təmin edir
   • Lavina effekti üçün vacibdir
""")

# 5. Matris seçimi
print("\n5. MATRİS SEÇİMİ:")
print("""
   [02 03 01 01] matrisi:
   • Bütün əmsallar kiçik (sürətli vurma üçün)
   • Determinant ≠ 0 (inversiya mümkün)
   • Hər bir çıxış baytı bütün giriş baytlarına bağlıdır
   • MDS xassəsinə malikdir; bir giriş fərqi mümkün qədər çox çıxış baytına yayılır (optimal branch number)
""")

## 🔑 Açar genişləndirmə (Key Expansion) (20 dəq)

In [ ]:
def rot_word(word):
    """
    RotWord: sözü bir bayt sola dövri sürüşdürür
    word: 4 baytlıq siyahı
    """
    return word[1:] + [word[0]]

def sub_word(word, sbox):
    """
    SubWord: hər bayta S-qutusu tətbiq edir
    """
    return [sub_bytes(b, sbox) for b in word]

# Rcon sabitləri (ilk 10)
RCON = [
    [0x01, 0x00, 0x00, 0x00],
    [0x02, 0x00, 0x00, 0x00],
    [0x04, 0x00, 0x00, 0x00],
    [0x08, 0x00, 0x00, 0x00],
    [0x10, 0x00, 0x00, 0x00],
    [0x20, 0x00, 0x00, 0x00],
    [0x40, 0x00, 0x00, 0x00],
    [0x80, 0x00, 0x00, 0x00],
    [0x1B, 0x00, 0x00, 0x00],
    [0x36, 0x00, 0x00, 0x00]
]

def xor_words(w1, w2):
    """
    İki sözü XOR edir
    """
    return [w1[i] ^ w2[i] for i in range(4)]

def key_expansion(key, sbox, key_size=128):
    """
    AES açar genişləndirmə
    key: 16/24/32 baytlıq açar (list/bytes)
    """
    if key_size not in (128, 192, 256):
        raise ValueError("key_size yalnız 128, 192 və ya 256 ola bilər")

    key = list(key)
    expected_len = key_size // 8
    if len(key) != expected_len:
        raise ValueError(
            f"{key_size}-bit AES üçün açar uzunluğu {expected_len} bayt olmalıdır"
        )

    if key_size == 128:
        nk = 4  # açar söz sayı
        nr = 10  # raund sayı
    elif key_size == 192:
        nk = 6
        nr = 12
    else:  # 256
        nk = 8
        nr = 14

    nb = 4  # blok söz sayı
    total_words = nb * (nr + 1)  # ümumi söz sayı

    # İlkin sözlər (əsas açar)
    w = []
    for i in range(0, len(key), 4):
        w.append(list(key[i:i+4]))

    # Qalan sözləri yarat
    for i in range(nk, total_words):
        temp = w[i-1].copy()

        if i % nk == 0:
            # RotWord -> SubWord -> Rcon
            temp = rot_word(temp)
            temp = sub_word(temp, sbox)
            temp = xor_words(temp, RCON[i // nk - 1])
        elif nk > 6 and i % nk == 4:
            # AES-256 üçün əlavə SubWord
            temp = sub_word(temp, sbox)

        w.append(xor_words(w[i-nk], temp))

    return w

# Test
print("\n" + "=" * 70)
print("🔑 AÇAR GENİŞLƏNDİRMƏ TESTİ")
print("=" * 70)

# FIPS 197 nümunə vektoru
key_bytes = [
    0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6,
    0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c
]
sbox = aes_sbox()
expanded = key_expansion(key_bytes, sbox, 128)

print("İlk 8 söz:")
for i in range(8):
    print(f"  w[{i}] = {[f'{x:02x}' for x in expanded[i]]}")


### ✍️ Çalışma 5: Açar genişləndirmə (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Açar sözləri:** Açar $K = 0x2B7E151628AED2A6ABF7158809CF4F3C$ üçün:
   * w[0], w[1], w[2], w[3] sözlərini tapın
   * w[4] sözünü hesablayın (i=4 üçün)

2. **Rcon sabitləri:** Rcon sabitlərinin rolu nədir? Niyə 0x1B, 0x36 kimi dəyərlər istifadə olunur?

3. **AES-192:** AES-192 üçün açar genişləndirməni implementasiya edin (nk=6, nr=12).

4. **AES-256 xüsusiyyəti:** AES-256-da niyə i % nk == 4 olduqda əlavə SubWord tətbiq olunur?

In [ ]:
# Çalışma 5 - Cavablar

print("📝 ÇALIŞMA 5 CAVABLARI")
print("=" * 80)

# 1. Açar sözləri
print("\n1. AÇAR SÖZLƏRİ:")
key_hex = "2B7E151628AED2A6ABF7158809CF4F3C"
key_bytes = bytes.fromhex(key_hex)
print(f"   Açar: {key_hex}")

w = []
for i in range(0, 16, 4):
    w.append(list(key_bytes[i:i+4]))

print(f"   w[0] = {[f'{x:02x}' for x in w[0]]}")
print(f"   w[1] = {[f'{x:02x}' for x in w[1]]}")
print(f"   w[2] = {[f'{x:02x}' for x in w[2]]}")
print(f"   w[3] = {[f'{x:02x}' for x in w[3]]}")

# w[4] hesabla
sbox = aes_sbox()
temp = rot_word(w[3].copy())
temp = sub_word(temp, sbox)
temp = xor_words(temp, RCON[0])
w4 = xor_words(w[0], temp)
print(f"   w[4] = {[f'{x:02x}' for x in w4]}")

# 2. Rcon sabitləri
print("\n2. RCON SABİTLƏRİ:")
print("""
   Rcon sabitləri (Round Constants):
   • Rcon[i] = [x^(i-1), 0, 0, 0] (GF(2⁸)-də)
   • x = 0x02
   • 0x1B, 0x36 kimi dəyərlər x^(i-1)-in nəticələridir
   • Rəqəmlərin təkrarlanmasının qarşısını alır
   • Açar genişləndirmədə simmetriyanı qırır
""")

# 3. AES-192 (nümunə)
print("\n3. AES-192:")
print("""
   AES-192:
   • nk = 6 (6 söz)
   • nr = 12 (12 raund)
   • Ümumi söz sayı = 4 * (12 + 1) = 52
   • Açar genişləndirmə eyni qayda ilə işləyir
""")

# 4. AES-256 xüsusiyyəti
print("\n4. AES-256 XÜSUSİYYƏTİ:")
print("""
   AES-256-da əlavə SubWord:
   • nk = 8 (8 söz)
   • i % 8 == 4 olduqda əlavə SubWord tətbiq olunur
   • Səbəb: daha çox qeyri-xəttilik və təhlükəsizlik
   • Bu, AES-256-nı AES-128/192-dən fərqləndirir
   • Xətti kriptoanalizə qarşı əlavə müdafiə
""")

## 🔐 Tam AES raundu (20 dəq)

In [ ]:
def add_round_key(state, round_key):
    """
    AddRoundKey: dövləti raund açarı ilə XOR edir
    round_key: 4x4 matris
    """
    result = [[0] * 4 for _ in range(4)]
    for i in range(4):
        for j in range(4):
            result[i][j] = state[i][j] ^ round_key[i][j]
    return result

def state_to_bytes(state):
    """
    4x4 matrisi 16 baytlıq siyahıya çevirir
    """
    result = []
    for j in range(4):  # sütun-sütun
        for i in range(4):
            result.append(state[i][j])
    return result

def bytes_to_state(data):
    """
    16 baytlıq siyahını 4x4 matrisə çevirir
    """
    if len(data) != 16:
        raise ValueError("AES state mütləq 16 bayt olmalıdır")

    state = [[0] * 4 for _ in range(4)]
    for i in range(4):
        for j in range(4):
            state[i][j] = data[j * 4 + i]
    return state

def round_words_to_state(expanded_keys, round_num):
    """
    Genişləndirilmiş açardan bir raund üçün 4 sözü götürüb
    4x4 round-key matrisinə çevirir.
    """
    start = 4 * round_num
    end = start + 4
    round_words = expanded_keys[start:end]

    if len(round_words) != 4:
        raise ValueError(f"Round {round_num} üçün 4 söz tapılmadı")

    round_key_bytes = [byte for word in round_words for byte in word]
    return bytes_to_state(round_key_bytes)

def aes_encrypt_block(plaintext, expanded_keys, sbox):
    """
    Bir bloku şifrləyir (AES-128/192/256)
    plaintext: 16 baytlıq siyahı
    expanded_keys: genişləndirilmiş açarlar (44/52/60 söz)
    """
    if len(plaintext) != 16:
        raise ValueError("AES yalnız 16 baytlıq bloklarla işləyir")

    if len(expanded_keys) not in (44, 52, 60):
        raise ValueError("expanded_keys uzunluğu 44, 52 və ya 60 söz olmalıdır")

    nr = len(expanded_keys) // 4 - 1  # 10, 12 və ya 14

    # İlkin dövlət
    state = bytes_to_state(plaintext)

    # İlkin AddRoundKey
    state = add_round_key(state, round_words_to_state(expanded_keys, 0))

    # Orta raundlar
    for round_num in range(1, nr):
        state = sub_bytes_state(state, sbox)
        state = shift_rows(state)
        state = mix_columns(state)
        state = add_round_key(state, round_words_to_state(expanded_keys, round_num))

    # Son raund (MixColumns yoxdur)
    state = sub_bytes_state(state, sbox)
    state = shift_rows(state)
    state = add_round_key(state, round_words_to_state(expanded_keys, nr))

    return state_to_bytes(state)

def aes_decrypt_block(ciphertext, expanded_keys, inv_sbox):
    """
    Bir bloku deşifrləyir (AES-128/192/256)
    """
    if len(ciphertext) != 16:
        raise ValueError("AES yalnız 16 baytlıq bloklarla işləyir")

    if len(expanded_keys) not in (44, 52, 60):
        raise ValueError("expanded_keys uzunluğu 44, 52 və ya 60 söz olmalıdır")

    nr = len(expanded_keys) // 4 - 1  # 10, 12 və ya 14
    state = bytes_to_state(ciphertext)

    # Son AddRoundKey
    state = add_round_key(state, round_words_to_state(expanded_keys, nr))

    # Orta raundların tərsi
    for round_num in range(nr - 1, 0, -1):
        state = inv_shift_rows(state)
        state = sub_bytes_state(state, inv_sbox)
        state = add_round_key(state, round_words_to_state(expanded_keys, round_num))
        state = inv_mix_columns(state)

    # İlkin raundun tərsi
    state = inv_shift_rows(state)
    state = sub_bytes_state(state, inv_sbox)
    state = add_round_key(state, round_words_to_state(expanded_keys, 0))

    return state_to_bytes(state)

# Test
print("\n" + "=" * 70)
print("🔐 AES BLOK ŞİFRLƏMƏ TESTİ")
print("=" * 70)

# FIPS 197 nümunə vektoru (AES-128)
plaintext_bytes = [
    0x32, 0x43, 0xF6, 0xA8, 0x88, 0x5A, 0x30, 0x8D,
    0x31, 0x31, 0x98, 0xA2, 0xE0, 0x37, 0x07, 0x34
]

expected_ciphertext = "3925841d02dc09fbdc118597196a0b32"

print("Açıq mətn:")
print(f"  {[f'{x:02x}' for x in plaintext_bytes]}")

ciphertext = aes_encrypt_block(plaintext_bytes, expanded, sbox)
ciphertext_hex = "".join(f"{x:02x}" for x in ciphertext)

print("\nŞifrəli mətn:")
print(f"  {[f'{x:02x}' for x in ciphertext]}")
print(f"  Gözlənilən hex: {expected_ciphertext}")
print(f"  Uyğundur? {ciphertext_hex == expected_ciphertext}")

decrypted = aes_decrypt_block(ciphertext, expanded, inv_sbox)
print("\nDeşifrələnmiş:")
print(f"  {[f'{x:02x}' for x in decrypted]}")
print(f"\n✅ Uğurlu? {plaintext_bytes == decrypted}")


### ✍️ Çalışma 6: Tam AES (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Test vektorları:** aes_encrypt_block funksiyasını test edin. NIST test vektorları ilə yoxlayın.

2. **Deşifrləmə:** aes_decrypt_block funksiyasını yazın (inv_sub_bytes, inv_shift_rows, inv_mix_columns istifadə edin).

3. **Şifrləmə/Deşifrləmə dövrü:** Şifrləmə və deşifrləmənin bir-birinin tərsi olduğunu yoxlayın.

4. **AES-192:** AES-192 üçün eyni funksiyaları yazın (12 raund, nk=6).

In [ ]:
# Çalışma 6 - Cavablar

print("📝 ÇALIŞMA 6 CAVABLARI")
print("=" * 80)

# 1-3. Test vektorları
print("\n1-3. TEST VEKTORLARI:")
test_vectors = {
    128: {
        "key": "000102030405060708090a0b0c0d0e0f",
        "pt": "00112233445566778899aabbccddeeff",
        "ct": "69c4e0d86a7b0430d8cdb78070b4c55a",
    },
    192: {
        "key": "000102030405060708090a0b0c0d0e0f1011121314151617",
        "pt": "00112233445566778899aabbccddeeff",
        "ct": "dda97ca4864cdfe06eaf70a0ec0d7191",
    },
    256: {
        "key": "000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f",
        "pt": "00112233445566778899aabbccddeeff",
        "ct": "8ea2b7ca516745bfeafc49904b496089",
    },
}

for key_size, vector in test_vectors.items():
    key = list(bytes.fromhex(vector["key"]))
    pt = list(bytes.fromhex(vector["pt"]))
    expected_ct = vector["ct"]

    expanded_test = key_expansion(key, sbox, key_size)
    ct = aes_encrypt_block(pt, expanded_test, sbox)
    ct_hex = "".join(f"{x:02x}" for x in ct)
    dec = aes_decrypt_block(ct, expanded_test, inv_sbox)
    dec_hex = "".join(f"{x:02x}" for x in dec)

    print(f"   AES-{key_size}:")
    print(f"      Şifrəli mətn     = {ct_hex}")
    print(f"      Gözlənilən       = {expected_ct}")
    print(f"      Şifrləmə düzgün? = {ct_hex == expected_ct}")
    print(f"      Deşifrləmə doğru? = {dec_hex == vector['pt']}")

# 4. AES-192 üçün qeyd
print("\n4. AES-192:")
print("""
   AES-192 üçün dəyişikliklər:
   • Açar: 24 bayt
   • nk = 6
   • nr = 12
   • Ümumi söz sayı = 4 * (12 + 1) = 52
   • Açar genişləndirmə 52 söz yaradır
   • Şifrləmə funksiyası eyni qayda ilə işləyir, sadəcə raund sayı 12-dir
""")

print("\n✅ AES-128 / AES-192 / AES-256 testləri uğurla keçdi!")


## 📦 PyCryptodome ilə AES (15 dəq)

In [ ]:
def aes_pycryptodome_demo():
    """
    PyCryptodome kitabxanası ilə AES-GCM istifadəsi.

    Qeyd: GCM AEAD rejimidir; həm məxfiliyi, həm də dəyişikliklərin
    aşkarlanmasını təmin edir. CBC istifadə edilərsə, ayrıca MAC lazımdır.
    """
    if not AES_AVAILABLE:
        print("PyCryptodome yüklü deyil")
        return

    print("\n" + "=" * 80)
    print("📦 PYCRYPTODOME İLƏ AES-128 GCM (AEAD)")
    print("=" * 80)

    # AES açarı (16 bayt = 128-bit)
    key = get_random_bytes(16)
    print(f"Açar (hex): {key.hex()}")

    # Məlumatı əvvəlcə str kimi saxla, sonra UTF-8 ilə kodlaşdır
    plaintext_text = "Salam, dünya! Bu AES test mesajıdır. Azərbaycan dili: ə, ü, ö, ç, ş, ğ"
    plaintext = plaintext_text.encode("utf-8")
    print(f"\nAçıq mətn ({len(plaintext)} bayt): {plaintext_text}")

    # GCM üçün nonce eyni açarla təkrar istifadə edilməməlidir.
    nonce = get_random_bytes(12)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)

    # Şifrləmə + autentifikasiya tag-i
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)
    print(f"\nNonce (hex): {nonce.hex()}")
    print(f"Şifrəli mətn (hex): {ciphertext.hex()}")
    print(f"Tag (hex): {tag.hex()}")

    # Deşifrləmə: plaintext qəbul edilməzdən əvvəl tag yoxlanılır.
    cipher_dec = AES.new(key, AES.MODE_GCM, nonce=nonce)
    try:
        decrypted = cipher_dec.decrypt_and_verify(ciphertext, tag)
    except ValueError:
        print("\n❌ Tag yoxlanışı uğursuz oldu: şifrəli mətn dəyişdirilmiş ola bilər")
        return key, nonce, ciphertext, tag

    decrypted_text = decrypted.decode("utf-8")
    print(f"\nDeşifrələnmiş: {decrypted_text}")
    print(f"\n✅ Uğurlu? {plaintext_text == decrypted_text}")

    # Kiçik yoxlama: ciphertext dəyişdirilərsə, GCM bunu aşkar edir.
    tampered = bytearray(ciphertext)
    if tampered:
        tampered[0] ^= 1
        cipher_check = AES.new(key, AES.MODE_GCM, nonce=nonce)
        try:
            cipher_check.decrypt_and_verify(bytes(tampered), tag)
            print("❌ Gözlənilməz: dəyişdirilmiş mətn qəbul edildi")
        except ValueError:
            print("✅ Dəyişdirilmiş ciphertext tag yoxlanışından keçmədi")

    return key, nonce, ciphertext, tag

if AES_AVAILABLE:
    aes_pycryptodome_demo()


### 11.1 AES-in müxtəlif iş rejimləri

In [ ]:
def aes_modes_demo():
    """
    AES-in müxtəlif iş rejimləri
    """
    if not AES_AVAILABLE:
        return

    print("\n" + "=" * 80)
    print("🔄 AES MÜXTƏLİF İŞ REJİMLƏRİ")
    print("=" * 80)

    key = get_random_bytes(16)
    plaintext = b"Qisa mesaj"

    # ECB rejimi təhlükəsiz deyil: eyni plaintext blokları eyni ciphertext bloklarına çevrilir.
    cipher_ecb = AES.new(key, AES.MODE_ECB)
    padded = pad(plaintext, AES.block_size)
    c_ecb = cipher_ecb.encrypt(padded)
    d_ecb = unpad(AES.new(key, AES.MODE_ECB).decrypt(c_ecb), AES.block_size)

    # CBC rejimi
    iv = get_random_bytes(16)
    cipher_cbc = AES.new(key, AES.MODE_CBC, iv)
    c_cbc = cipher_cbc.encrypt(padded)
    d_cbc = unpad(AES.new(key, AES.MODE_CBC, iv).decrypt(c_cbc), AES.block_size)

    # CTR rejimi (padding lazım deyil)
    cipher_ctr = AES.new(key, AES.MODE_CTR)
    c_ctr = cipher_ctr.encrypt(plaintext)
    nonce_ctr = cipher_ctr.nonce
    d_ctr = AES.new(key, AES.MODE_CTR, nonce=nonce_ctr).decrypt(c_ctr)

    # GCM rejimi (şifrləmə + autentifikasiya).
    # Vacib: AES-GCM-də eyni açarla nonce/IV heç vaxt təkrarlanmamalıdır.
    cipher_gcm = AES.new(key, AES.MODE_GCM)
    c_gcm, tag = cipher_gcm.encrypt_and_digest(plaintext)
    nonce_gcm = cipher_gcm.nonce
    cipher_gcm_dec = AES.new(key, AES.MODE_GCM, nonce=nonce_gcm)
    d_gcm = cipher_gcm_dec.decrypt_and_verify(c_gcm, tag)

    print(f"ECB şifrəli: {c_ecb.hex()}")
    print(f"CBC şifrəli: {c_cbc.hex()}")
    print(f"CTR şifrəli: {c_ctr.hex()}")
    print(f"GCM şifrəli: {c_gcm.hex()}")
    print(f"GCM tag: {tag.hex()}")

    print("\nDoğrulama:")
    print(f"   ECB deşifrləmə doğru? {d_ecb == plaintext}")
    print(f"   CBC deşifrləmə doğru? {d_cbc == plaintext}")
    print(f"   CTR deşifrləmə doğru? {d_ctr == plaintext}")
    print(f"   GCM deşifrləmə + verify doğru? {d_gcm == plaintext}")

    print("\n📌 QEYDLƏR:")
    print("   • CTR rejimində padding lazım deyil və paralel işləyə bilər")
    print("   • GCM rejimi həm şifrləmə, həm də autentifikasiya təmin edir")
    print("   • AES-GCM-də eyni açarla nonce/IV heç vaxt təkrarlanmamalıdır; nonce-u ciphertext/tag ilə birlikdə saxlayın")
    print("   • ECB rejimi təhlükəsiz DEYİL - eyni bloklar eyni şifrələnir")

if AES_AVAILABLE:
    aes_modes_demo()


### ✍️ Çalışma 7: PyCryptodome ilə AES (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Fayl şifrləmə:** PyCryptodome istifadə edərək bir faylı AES-128-CBC ilə şifrləyin.
   * Faylın adını və şifrəni istifadəçidən alın
   * PBKDF2 istifadə edərək şifrədən açar yaradın
   * IV təsadüfi olsun

2. **Deşifrləmə:** Eyni açar və IV ilə faylı deşifrləyin.

3. **CTR rejimi:** CTR rejimində şifrləmə aparın. CBC ilə müqayisə edin (sürət, padding).

4. **GCM rejimi:** GCM rejiminin üstünlükləri nələrdir? Autentifikasiya niyə vacibdir?

In [ ]:
# Çalışma 7 - Cavablar

print("📝 ÇALIŞMA 7 CAVABLARI")
print("=" * 80)

if AES_AVAILABLE:
    # 1-2. Fayl şifrləmə (nümunə kod)
    print("\n1-2. FAYL ŞİFRLƏMƏ NÜMUNƏSİ:")
    print("""
    from Crypto.Hash import SHA512

    def encrypt_file(filename, password):
        salt = get_random_bytes(16)
        key = PBKDF2(password, salt, 32, count=1000000, hmac_hash_module=SHA512)
        iv = get_random_bytes(16)

        with open(filename, 'rb') as f:
            data = f.read()

        cipher = AES.new(key, AES.MODE_CBC, iv)
        padded = pad(data, AES.block_size)
        encrypted = cipher.encrypt(padded)

        with open(filename + '.aes', 'wb') as f:
            f.write(salt + iv + encrypted)

    def decrypt_file(filename, password):
        with open(filename, 'rb') as f:
            blob = f.read()

        salt = blob[:16]
        iv = blob[16:32]
        encrypted = blob[32:]
        key = PBKDF2(password, salt, 32, count=1000000, hmac_hash_module=SHA512)

        cipher = AES.new(key, AES.MODE_CBC, iv)
        padded = cipher.decrypt(encrypted)
        return unpad(padded, AES.block_size)
    """)

    # 3. CTR rejimi
    print("\n3. CTR REJİMİ:")
    print("""
    CTR üstünlükləri:
    • Padding lazım deyil
    • Paralel şifrləmə mümkün
    • Random access (istənilən bloku deşifrləmək olar)

    CBC ilə müqayisə:
    • CBC daha geniş yayılıb
    • CTR daha sürətli (paralel olduğu üçün)
    • Hər ikisi təhlükəsizdir (düzgün istifadə edildikdə)
    """)

    # 4. GCM rejimi
    print("\n4. GCM REJİMİ:")
    print("""
    GCM (Galois/Counter Mode):
    • AEAD (Authenticated Encryption with Associated Data)
    • Həm şifrləmə, həm də autentifikasiya təmin edir
    • MAC ayrıca hesablamağa ehtiyac yoxdur
    • Məlumat bütövlüyünü təmin edir
    • TLS 1.2 və 1.3-də geniş istifadə olunur

    Autentifikasiya vacibdir, çünki:
    • Şifrəli mətn dəyişdirilə bilər (active attacker)
    • MAC olmadan dəyişiklik aşkar edilmir
    • Seçilmiş şifrəli mətn hücumlarına qarşı qoruyur

    Praktik qeyd:
    • Real sistemlərdə CBC əvəzinə çox vaxt GCM seçilir
    • CBC istifadə edilərsə, ayrıca autentifikasiya (məsələn, HMAC) əlavə olunmalıdır
    """)


## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def aes_lab_menu():
    """
    AES laboratoriyası interaktiv menyu
    """
    while True:
        print("\n" + "=" * 70)
        print("🔐 AES LABORATORİYASI - Məşğələ 10")
        print("=" * 70)
        print("1. 🧮 GF(2⁸) əməliyyatları test et")
        print("2. 🔢 S-qutusu testi")
        print("3. 🔄 ShiftRows testi")
        print("4. 🔀 MixColumns testi")
        print("5. 🔑 Açar genişləndirmə")
        print("6. 🔐 AES-128 blok şifrləmə")
        print("7. 📦 PyCryptodome ilə AES (CBC)")
        print("8. 📁 Fayl şifrləmə (AES-256)")
        print("9. 🔄 AES rejimlərini müqayisə et")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == '1':
            a = int(input("a (hex, məsələn, 57): ") or "57", 16)
            b = int(input("b (hex, məsələn, 13): ") or "13", 16)
            print(f"gf_multiply(0x{a:02x}, 0x{b:02x}) = 0x{gf_multiply(a, b):02x}")
            print(f"xtime(0x{a:02x}) = 0x{xtime(a):02x}")
            print(f"inverse(0x{a:02x}) = 0x{gf_inverse(a):02x}")

        elif choice == '2':
            sbox = aes_sbox()
            inv_sbox = aes_inv_sbox()
            val = int(input("Bayt dəyəri (hex): ") or "53", 16)
            result = sub_bytes(val, sbox)
            inv_result = sub_bytes(result, inv_sbox)
            print(f"S-box(0x{val:02x}) = 0x{result:02x}")
            print(f"InvS-box(0x{result:02x}) = 0x{inv_result:02x}")

        elif choice == '3':
            state = [
                [1,2,3,4],
                [5,6,7,8],
                [9,10,11,12],
                [13,14,15,16]
            ]
            print("Orijinal:")
            for row in state:
                print(f"  {row}")

            shifted = shift_rows(state)
            print("\nShiftRows:")
            for row in shifted:
                print(f"  {row}")

            inv_shifted = inv_shift_rows(shifted)
            print("\nInvShiftRows:")
            for row in inv_shifted:
                print(f"  {row}")

        elif choice == '4':
            col = [0x87, 0x6E, 0x46, 0xA6]
            print(f"Sütun: {[f'{x:02x}' for x in col]}")

            s0, s1, s2, s3 = col
            res0 = gf_multiply(0x02, s0) ^ gf_multiply(0x03, s1) ^ s2 ^ s3
            res1 = s0 ^ gf_multiply(0x02, s1) ^ gf_multiply(0x03, s2) ^ s3
            res2 = s0 ^ s1 ^ gf_multiply(0x02, s2) ^ gf_multiply(0x03, s3)
            res3 = gf_multiply(0x03, s0) ^ s1 ^ s2 ^ gf_multiply(0x02, s3)

            print(f"Nəticə: [{res0:02x}, {res1:02x}, {res2:02x}, {res3:02x}]")

        elif choice == '5':
            key_hex = input("Açar (32 hex simvol): ") or "2B7E151628AED2A6ABF7158809CF4F3C"
            key_bytes = bytes.fromhex(key_hex)
            key_list = list(key_bytes)

            sbox = aes_sbox()
            expanded = key_expansion(key_list, sbox, 128)

            for i in range(11):
                words = expanded[i*4:(i+1)*4]
                print(f"Round {i:2d}: {[f'{x:02x}' for word in words for x in word]}")

        elif choice == '6':
            # NIST test vektoru
            key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6,
                   0xab, 0xf7, 0x15, 0x88, 0x09, 0xcf, 0x4f, 0x3c]
            plain = [0x32, 0x43, 0xf6, 0xa8, 0x88, 0x5a, 0x30, 0x8d,
                     0x31, 0x31, 0x98, 0xa2, 0xe0, 0x37, 0x07, 0x34]

            sbox = aes_sbox()
            inv_sbox = aes_inv_sbox()
            expanded = key_expansion(key, sbox, 128)
            cipher = aes_encrypt_block(plain, expanded, sbox)
            decrypted = aes_decrypt_block(cipher, expanded, inv_sbox)

            print("Açıq mətn:", [f"{x:02x}" for x in plain])
            print("Şifrəli:   ", [f"{x:02x}" for x in cipher])
            print("Deşifrə:   ", [f"{x:02x}" for x in decrypted])
            print(f"\n✅ Uğurlu? {plain == decrypted}")

        elif choice == '7' and AES_AVAILABLE:
            text = input("Mətn: ")
            key = get_random_bytes(16)
            iv = get_random_bytes(16)

            cipher = AES.new(key, AES.MODE_CBC, iv)
            padded = pad(text.encode(), AES.block_size)
            encrypted = cipher.encrypt(padded)

            print(f"Açar: {key.hex()}")
            print(f"IV: {iv.hex()}")
            print(f"Şifrəli: {encrypted.hex()}")

            cipher2 = AES.new(key, AES.MODE_CBC, iv)
            decrypted = unpad(cipher2.decrypt(encrypted), AES.block_size)
            print(f"Deşifrə: {decrypted.decode()}")

        elif choice == '8' and AES_AVAILABLE:
            filename = input("Fayl adı: ")
            password = input("Şifrə: ").encode()

            # Şifrədən açar yarat (PBKDF2)
            salt = get_random_bytes(16)
            key = PBKDF2(password, salt, 32, count=100000)  # 256-bit açar

            try:
                with open(filename, 'rb') as f:
                    data = f.read()

                iv = get_random_bytes(16)
                cipher = AES.new(key, AES.MODE_CBC, iv)
                padded = pad(data, AES.block_size)
                encrypted = cipher.encrypt(padded)

                # Salt + IV + şifrəli məlumat
                with open(filename + '.aes', 'wb') as f:
                    f.write(salt + iv + encrypted)

                print(f"✅ Fayl şifrləndi: {filename}.aes")
            except Exception as e:
                print(f"❌ Xəta: {e}")

        elif choice == '9' and AES_AVAILABLE:
            aes_modes_demo()

        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim")

# Proqramı işə sal (istəyə bağlı)
# aes_lab_menu()

## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: AES paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
aes_package/
├── __init__.py
├── gf.py                 # GF(2⁸) əməliyyatları (gf_multiply, xtime, gf_inverse)
├── sbox.py               # AES S-qutusu, inv_sbox, sub_bytes, sub_word
├── shiftrows.py          # ShiftRows, inv_shift_rows
├── mixcolumns.py         # MixColumns, inv_mix_columns
├── key_expansion.py      # Açar genişləndirmə (AES-128, 192, 256)
├── aes.py                # Tam AES şifrləmə/deşifrləmə
└── main.py               # Bütün funksiyaları birləşdirən interaktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin
* NIST test vektorları ilə düzgünlüyü yoxlayın

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **S-qutusunun riyazi hesablanması:** AES S-qutusunu riyazi olaraq hesablayan funksiya yazın (axtarış cədvəli əvəzinə).
   * GF(2⁸)-də tərs element
   * Affin transformasiya

2. **AES-128 tam implementasiya:** AES-128-in tam implementasiyasını yazın. NIST test vektorları ilə yoxlayın.
   * FIPS 197 əlavə B və C-də verilən şifrələmə nümunəsi və test vektorları
   * Bütün raundlar üçün ara dəyərləri yoxlayın

3. **AES-256:** AES-256 ilə bir faylı şifrləyin və deşifrləyin.
   * PBKDF2 istifadə edərək şifrədən açar yaradın
   * Salt və IV-i faylla birlikdə saxlayın

4. **CTR rejimi:** CTR rejimində AES implementasiya edin (paralel şifrləmə üçün).
   * Sayğacın düzgün artırıldığından əmin olun
   * Random access xüsusiyyətini nümayiş etdirin


### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **Rijndael-in dizaynerləri:** Joan Daemen və Vincent Rijmen kimdir? Rijndael-in dizayn fəlsəfəsi nədir?
   * Niyə məhz bu strukturu seçdilər?
   * DES-dən nə ilə fərqlənir?

2. **AES finalçıları:** NIST AES müsabiqəsində iştirak edən digər finalçılar (MARS, RC6, Serpent, Twofish) haqqında məlumat toplayın.
   * Hər birinin üstünlükləri və çatışmazlıqları
   * Niyə Rijndael seçildi?

3. **AES kriptoanalizi:** AES-256-ya qarşı məlum olan ən yaxşı nəzəri hücum hansıdır?
   * Biclique hücumu (2011)
   * Pratik hücumlar varmı?
   * Mürəkkəblik nə qədərdir?

4. **Post-kvant dövr:** Post-kvant dövrdə AES-in rolu nə olacaq?
   * Kvant kompüterlər AES-ə təhlükə yaradırmı?
   * Grover alqoritmi və təsiri
   * Açar ölçüsünün iki qat artırılması kifayətdirmi?

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, sxemlər və ya diaqramlar əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>$GF(2^8)$ arifmetikası</b> — xtime, gf_multiply, $m(x) = x^8 + x^4 + x^3 + x + 1$</li>
    <li>✅ <b>SubBytes</b> — S-qutusu, qeyri-xətti əvəzetmə (tərs element + affin transformasiya)</li>
    <li>✅ <b>ShiftRows</b> — sətirlərin dövri sürüşdürülməsi (0,1,2,3 bayt)</li>
    <li>✅ <b>MixColumns</b> — sütunların qarışdırılması, matris vurması</li>
    <li>✅ <b>AddRoundKey</b> — XOR ilə açarın əlavə edilməsi</li>
    <li>✅ <b>Key Expansion</b> — raund açarlarının yaradılması, RotWord, SubWord, Rcon</li>
    <li>✅ <b>AES rejimləri</b> — ECB, CBC, CTR, GCM</li>
</ul>
<p><i>AES müasir kriptoqrafiyanın ən mühüm alqoritmlərindən biridir və dünyada ən geniş istifadə olunan simmetrik şifrləmə standartıdır.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. DES ilə AES arasında əsas fərqlər nələrdir? (Feystel vs SPN, blok ölçüsü, açar uzunluğu)
3. Niyə MixColumns ən mürəkkəb mərhələdir? O olmasa nə olardı?
4. Hansı AES rejimi daha təhlükəsizdir? Niyə? (ECB-nin problemi nədir?)
5. S-qutusunun qurulmasında affin transformasiyanın rolu nədir?
6. AES-256 niyə AES-128-dən təhlükəsizdir? Praktik fərq nədir?
7. GCM rejimi niyə müasir protokollarda (TLS 1.3) geniş istifadə olunur?

## 📚 Əlavə resurslar

* 📘 **FIPS 197: AES standartı:** [https://csrc.nist.gov/pubs/fips/197/final](https://csrc.nist.gov/pubs/fips/197/final) — 2023 yeniləməsi redaksiya xarakterlidir; AES alqoritminin özü dəyişməyib.
* 📙 **PyCryptodome sənədləşməsi:** [https://pycryptodome.readthedocs.io/](https://pycryptodome.readthedocs.io/)
* 📗 **Daemen, J., Rijmen, V. (2002). "The Design of Rijndael"** — AES-in dizaynerlərindən əsas kitab
* 📕 **AES test vektorları:** [https://csrc.nist.gov/projects/cryptographic-algorithm-validation-program/block-ciphers](https://csrc.nist.gov/projects/cryptographic-algorithm-validation-program/block-ciphers)
* 📘 **AES animasiyası:** [http://www.formaestudio.com/rijndaelinspector/](http://www.formaestudio.com/rijndaelinspector/)
* 📙 **Serpent, Twofish, RC6, MARS haqqında:** [https://en.wikipedia.org/wiki/Advanced_Encryption_Standard_process](https://en.wikipedia.org/wiki/Advanced_Encryption_Standard_process)
* 📗 **AES kriptoanalizi (Biclique attack):** [https://en.wikipedia.org/wiki/Biclique_attack](https://en.wikipedia.org/wiki/Biclique_attack)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 10 tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔐 <b>AES - müasir kriptoqrafiyanın qızıl standartı!</b></p>
</div>